# 🤖📄 RAG sobre PDF con un OCR **open-source** en tu propia GPU

**Curso práctico · ~90 minutos · Google Colab (GPU) + BigQuery + Unlimited-OCR**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/noelserdna/colab-gcp-ia/blob/main/curso_rag_pdf_ocr_opensource.ipynb)

Seguimos en **Peñalara Seguros, S.A.**, con el mismo objetivo del curso anterior: responder *"¿esto me lo cubre la póliza?"* citando la página y la cláusula. Pero cambiamos **una pieza clave**: en vez de mandar los PDF a la API de **Document AI** de Google, extraemos el texto con un **modelo open-source que corre en la GPU de este Colab**: [`baidu/Unlimited-OCR`](https://huggingface.co/baidu/Unlimited-OCR) (licencia MIT, sucesor de DeepSeek-OCR).

> ⚠️ **Este cuaderno necesita GPU.** *Entorno de ejecución → Cambiar tipo de entorno → GPU*. Una **T4** (gratis) sirve; una **L4/A100** va bastante más rápida. Sin GPU, el bloque de OCR no puede ejecutarse.

> **Es la tercera parte de una serie.** Da por vistos los fundamentos de RAG (embedding, `task_type`, `VECTOR_SEARCH`, el patrón recuperar→aumentar→generar). Si vienes de nuevas, el **🎒 Kit de supervivencia** de abajo te pone al día en dos minutos.


## 🎒 Kit de supervivencia (si es tu primera vez con RAG)

Cinco ideas, treinta segundos cada una:

- **Embedding** — convertir un texto en un **vector** de números que captura su significado. Textos que quieren decir lo mismo tienen vectores parecidos, aunque no compartan palabras.
- **Distancia / similitud** — si cada texto es un punto, los parecidos caen **cerca**. Buscar = encontrar los vecinos más próximos a la pregunta (distancia coseno).
- **Chunk** — un documento es demasiado grande para vectorizarlo entero: se **trocea** en fragmentos y se vectoriza cada uno. Cómo se trocea es decisivo.
- **`task_type`** — al pedir un embedding indicas su uso: **`RETRIEVAL_DOCUMENT`** para los documentos, **`RETRIEVAL_QUERY`** para la pregunta.
- **RAG** — **(1) recuperar** los fragmentos relevantes, **(2) aumentar** el prompt con ellos, **(3) generar** la respuesta usando solo ese contexto, con **citas**.

> Para el detalle a fondo están los dos cursos anteriores ([emails](https://colab.research.google.com/github/noelserdna/colab-gcp-ia/blob/main/curso_rag_emails_bigquery_v2.ipynb) y [PDF con Document AI](https://colab.research.google.com/github/noelserdna/colab-gcp-ia/blob/main/curso_rag_pdf_polizas_bigquery.ipynb)). Aquí vamos al grano.


---
## ⚖️ La pregunta que abre el curso: ¿por qué un OCR propio en vez de Document AI?

El curso anterior resolvía la extracción con **Document AI Layout Parser**: una API gestionada de Google, excelente, que por debajo hacía el troceado por ti. Funciona de maravilla… pero tiene tres peajes que a veces importan mucho:

| | Document AI (curso anterior) | **Unlimited-OCR (este curso)** |
|---|---|---|
| Dónde se procesa el PDF | en los servidores de Google (**el documento sale de tu máquina**) | **en la GPU de este Colab** (el documento no sale) |
| Coste | $10 / 1.000 páginas | **gratis** (solo el tiempo de GPU) |
| Dependencia | API propietaria de un proveedor | modelo abierto (**MIT**), lo puedes desplegar donde quieras |
| Chunking | te lo daba **hecho** (layout-aware) | **lo haces tú** sobre el markdown del modelo |
| Infra | ninguna (llamada SQL) | cargar un modelo de 6,7 GB en una GPU |

### 🧠 El argumento de fondo: soberanía del dato

En seguros —igual que en banca, salud o legal— los documentos llevan **datos personales**. Mandarlos a una API de terceros es, muchas veces, un problema de cumplimiento (RGPD, secreto, contratos que prohíben sacar el dato del perímetro). Un OCR que corre **dentro de tu infraestructura** cambia esa conversación por completo: el PDF **nunca sale**.

> 🎓 **La tesis de este curso:** el *resultado* que buscamos es idéntico al del curso anterior (chunks con estructura → embeddings → búsqueda → RAG con citas). Lo que cambia es **quién hace la extracción y dónde**. Y ese "dónde" es una decisión de arquitectura, no de comodidad.


### 📋 Requisitos

- **GPU activada** en Colab (T4 gratis vale; L4/A100 mejor).
- Un **proyecto GCP con BigQuery y Vertex AI** habilitados (para embeddings y generación — igual que en los cursos anteriores).
- **No** necesitas Document AI, ni Cloud Storage, ni facturación por página: la extracción es gratis y local.

### Agenda

| # | Bloque | ⏱️ |
|---|--------|----|
| 0 | Setup (GPU + BigQuery + Vertex) | 8 min |
| 1 | Fabricar las pólizas en PDF | 5 min |
| 2 | **Unlimited-OCR: del PDF (imagen) a markdown** | 20 min |
| 3 | **De markdown a chunks con estructura** (el chunking es tuyo) | 15 min |
| 4 | El contraste: OCR propio vs. Document AI | 5 min |
| 5 | Embeddings y búsqueda | 8 min |
| 6 | RAG con citas verificables | 12 min |
| 7 | Versionado de documentos | 8 min |
| 8 | Evaluación, ejercicios y limpieza | 9 min |


---
# 0 · Setup ⏱️ ~8 min

▶️ **Qué hace esta celda:** instala todo. Frente al curso anterior **desaparecen** `google-cloud-storage` y Document AI, y **aparecen** las librerías del modelo: `transformers` (fijado a la versión que pide el modelo), `pymupdf` (para pasar el PDF a imágenes) y utilidades. **No tocamos `torch`**: usamos el que Colab ya trae con CUDA (reinstalarlo obligaría a reiniciar el entorno).

In [ ]:
# NB: no instalamos torch (usamos el de Colab) ni flash-attn (el modelo no lo necesita).
%pip install --quiet google-cloud-bigquery google-genai pandas db-dtypes reportlab \
    "transformers==4.57.1" pymupdf einops addict easydict
print("✅ Dependencias instaladas")

▶️ **Qué hace esta celda:** comprueba que **hay GPU** antes de seguir. Si esto falla, ve a *Entorno de ejecución → Cambiar tipo de entorno → GPU* y reejecuta.

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError("❌ No hay GPU. Entorno de ejecución → Cambiar tipo de entorno → GPU (T4). "
                       "Sin GPU no se puede correr el OCR.")
gpu = torch.cuda.get_device_properties(0)
print(f"✅ GPU: {gpu.name} · {gpu.total_memory/1e9:.0f} GB")
if "T4" in gpu.name:
    print("ℹ️  Es una T4: funciona, pero el OCR irá lento (Turing no acelera bfloat16). "
          "Si puedes, elige L4 o A100.")

▶️ **Qué hace esta celda:** te autentica y fija proyecto, región y dataset. Igual que en los cursos anteriores. La región debe ser **`US`** o **`EU`** (lo exige `ML.GENERATE_EMBEDDING`).

In [ ]:
from google.colab import auth
auth.authenticate_user()
print("✅ Autenticado")

# 👇 EDITA ESTO con tu proyecto
PROJECT_ID = "tu-proyecto-gcp"  # @param {type:"string"}
LOCATION   = "US"               # US o EU (obligatorio para ML.GENERATE_EMBEDDING)
DATASET    = "rag_polizas_ocr"

!gcloud config set project {PROJECT_ID} --quiet
!gcloud services enable bigquery.googleapis.com bigqueryconnection.googleapis.com \
    aiplatform.googleapis.com --quiet
print("✅ APIs habilitadas")

▶️ **Qué hace esta celda:** crea el cliente de BigQuery y el dataset.

In [ ]:
from google.cloud import bigquery

client = bigquery.Client(project=PROJECT_ID, location=LOCATION)
ds = bigquery.Dataset(f"{PROJECT_ID}.{DATASET}")
ds.location = LOCATION
client.create_dataset(ds, exists_ok=True)
print(f"✅ Dataset listo: {PROJECT_ID}.{DATASET}")

▶️ **Qué hace esta celda:** la conexión de BigQuery a Vertex AI y su permiso. **Más simple que en el curso anterior**: como ya no usamos Document AI ni Cloud Storage, la service account solo necesita **un** rol (`aiplatform.user`) para los embeddings — antes hacían falta tres.

In [ ]:
import json, subprocess, time

CONN_ID = "vertex_conn"
!bq mk --connection --location={LOCATION} --project_id={PROJECT_ID} \
    --connection_type=CLOUD_RESOURCE {CONN_ID} 2>/dev/null || echo "(la conexión ya existía)"

out = subprocess.run(
    ["bq", "show", "--connection", "--format=json", f"{PROJECT_ID}.{LOCATION}.{CONN_ID}"],
    capture_output=True, text=True)
if out.returncode != 0:
    raise RuntimeError(f"'bq show' falló:\n{out.stderr}")
sa = json.loads(out.stdout[out.stdout.find("{"):])["cloudResource"]["serviceAccountId"]
print("Service account de la conexión:", sa)

r = subprocess.run(
    ["gcloud", "projects", "add-iam-policy-binding", PROJECT_ID,
     f"--member=serviceAccount:{sa}", "--role=roles/aiplatform.user",
     "--condition=None", "--quiet"], capture_output=True, text=True)
print("✅ rol aiplatform.user concedido" if r.returncode == 0 else "❌ " + r.stderr[:150])

> 🚩 **CHECKPOINT 1** — Deberías ver la GPU detectada, `✅ Dataset listo` y el rol concedido. Si el rol falla, suele ser permisos IAM insuficientes en tu usuario sobre el proyecto.

---
# 1 · Fabricamos las pólizas en PDF ⏱️ ~5 min

Idéntico al curso anterior: generamos tres condicionados de seguro con `reportlab`, con su numeración jerárquica, sus tablas y —lo importante— la sección **4. EXCLUSIONES** con texto peligrosamente parecido al de **3. COBERTURAS**. Ese es el caso que pondrá a prueba el RAG.

▶️ **Qué hace esta celda:** estilos y plantilla del documento (incluye la *letra pequeña* y el pie *"Página X"*).

In [ ]:
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import cm
from reportlab.lib import colors
from reportlab.platypus import (BaseDocTemplate, PageTemplate, Frame, Paragraph,
                                Spacer, Table, TableStyle, PageBreak)
from reportlab.lib.enums import TA_JUSTIFY, TA_CENTER

ASEGURADORA = "Peñalara Seguros, S.A."

ss = getSampleStyleSheet()
H1 = ParagraphStyle("H1x", parent=ss["Heading1"], fontSize=15, spaceAfter=10,
                    textColor=colors.HexColor("#1a3d5c"))
H2 = ParagraphStyle("H2x", parent=ss["Heading2"], fontSize=12, spaceBefore=10,
                    spaceAfter=6, textColor=colors.HexColor("#2c5f8a"))
H3 = ParagraphStyle("H3x", parent=ss["Heading3"], fontSize=10.5, spaceBefore=8,
                    spaceAfter=4, textColor=colors.HexColor("#444444"))
BODY = ParagraphStyle("BODYx", parent=ss["BodyText"], fontSize=9.5, leading=13,
                      alignment=TA_JUSTIFY, spaceAfter=5)
# 👇 la famosa "letra pequeña": legalmente válida, visualmente hostil
SMALL = ParagraphStyle("SMALLx", parent=BODY, fontSize=6.5, leading=8.5,
                       textColor=colors.HexColor("#555555"))
TITLE = ParagraphStyle("TITLEx", parent=ss["Title"], fontSize=22,
                       textColor=colors.HexColor("#1a3d5c"))
CENTER = ParagraphStyle("CENTERx", parent=BODY, alignment=TA_CENTER)

def _tabla(data, col_widths):
    t = Table(data, colWidths=col_widths, repeatRows=1)
    t.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#1a3d5c")),
        ("TEXTCOLOR", (0, 0), (-1, 0), colors.white),
        ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),
        ("FONTSIZE", (0, 0), (-1, -1), 8),
        ("GRID", (0, 0), (-1, -1), 0.4, colors.grey),
        ("ROWBACKGROUNDS", (0, 1), (-1, -1), [colors.white, colors.HexColor("#eef3f8")]),
    ]))
    return t

class PolizaDoc(BaseDocTemplate):
    """Documento con encabezado y pie 'Página X de Y' en cada página."""
    def __init__(self, filename, producto, codigo, **kw):
        super().__init__(filename, pagesize=A4, **kw)
        self.producto, self.codigo = producto, codigo
        frame = Frame(2.2*cm, 2.2*cm, A4[0]-4.4*cm, A4[1]-4.4*cm, id="n")
        self.addPageTemplates([PageTemplate(id="all", frames=[frame], onPage=self._decorar)])

    def _decorar(self, canvas, doc):
        canvas.saveState()
        canvas.setFont("Helvetica", 7)
        canvas.setFillColor(colors.HexColor("#777777"))
        canvas.drawString(2.2*cm, A4[1]-1.5*cm, f"{ASEGURADORA} · {self.producto}")
        canvas.drawRightString(A4[0]-2.2*cm, A4[1]-1.5*cm, f"Condicionado {self.codigo}")
        canvas.line(2.2*cm, A4[1]-1.65*cm, A4[0]-2.2*cm, A4[1]-1.65*cm)
        canvas.line(2.2*cm, 1.9*cm, A4[0]-2.2*cm, 1.9*cm)
        canvas.drawCentredString(A4[0]/2.0, 1.4*cm, f"Página {doc.page}")
        canvas.restoreState()

print("✅ Estilos y plantilla definidos")

▶️ **Qué hace esta celda:** la función que arma cada póliza sección a sección.

In [ ]:
def construir_poliza(path, producto, codigo, version, coberturas, exclusiones, franquicias):
    story = []
    # ── Portada ──
    story += [Spacer(1, 4*cm), Paragraph(ASEGURADORA, CENTER), Spacer(1, 1*cm),
              Paragraph(producto, TITLE), Spacer(1, 0.6*cm),
              Paragraph("Condiciones Generales", CENTER), Spacer(1, 0.3*cm),
              Paragraph(f"Código de condicionado: <b>{codigo}</b> · Versión {version}", CENTER),
              PageBreak()]

    # ── 1. Definiciones ──
    story += [Paragraph("1. DEFINICIONES", H1),
              Paragraph("A efectos del presente contrato, se entiende por:", BODY)]
    for num, term, txt in [
        ("1.1", "Asegurado", "Persona física o jurídica titular del interés asegurado y sobre la que recaen las consecuencias económicas del siniestro."),
        ("1.2", "Tomador", "Persona que suscribe el contrato con el Asegurador y a quien corresponden las obligaciones derivadas del mismo."),
        ("1.3", "Siniestro", "Todo hecho cuyas consecuencias estén total o parcialmente cubiertas por las garantías de esta póliza."),
        ("1.4", "Franquicia", "Cantidad que queda a cargo del Asegurado en cada siniestro y que se deduce de la indemnización."),
        ("1.5", "Suma asegurada", "Límite máximo de indemnización por siniestro y anualidad de seguro."),
    ]:
        story += [Paragraph(f"{num} {term}", H3), Paragraph(txt, BODY)]

    # ── 2. Objeto ──
    story += [PageBreak(), Paragraph("2. OBJETO DEL SEGURO", H1),
              Paragraph(f"El Asegurador garantiza, dentro de los límites del presente condicionado, "
                        f"las consecuencias económicas de los riesgos descritos en la sección 3, hasta "
                        f"las sumas fijadas en las Condiciones Particulares de la póliza {producto}.", BODY),
              Paragraph("2.1 Ámbito territorial", H3),
              Paragraph("Las garantías surten efecto en el territorio español, salvo indicación expresa "
                        "en contrario en las Condiciones Particulares.", BODY),
              Paragraph("2.2 Ámbito temporal", H3),
              Paragraph("Quedan cubiertos los siniestros ocurridos durante la vigencia de la póliza y "
                        "declarados conforme a los plazos de la sección 6.", BODY)]

    # ── 3. Coberturas (CON TABLA) ──
    story += [PageBreak(), Paragraph("3. COBERTURAS", H1),
              Paragraph("Quedan cubiertas las siguientes garantías, con los límites indicados:", BODY),
              Spacer(1, 0.3*cm),
              _tabla([["Garantía", "Límite por siniestro", "Franquicia"]] + coberturas,
                     [7.5*cm, 4.5*cm, 3.5*cm]), Spacer(1, 0.4*cm)]
    for i, (gar, lim, fr) in enumerate(coberturas, start=1):
        story += [Paragraph(f"3.{i} {gar}", H3),
                  Paragraph(f"Se garantiza el pago de la indemnización por los daños directos "
                            f"ocasionados por {gar.lower()}, hasta el límite de {lim} por siniestro, "
                            f"con una franquicia de {fr}. La cobertura opera siempre que el hecho "
                            f"causante sea súbito, accidental e imprevisto para el Asegurado.", BODY)]

    # ── 4. Exclusiones (🎯 el gotcha) ──
    story += [PageBreak(), Paragraph("4. EXCLUSIONES", H1),
              Paragraph("<b>Con carácter general, y salvo pacto expreso en contrario, quedan "
                        "EXCLUIDOS de toda cobertura:</b>", BODY)]
    for i, (titulo, items) in enumerate(exclusiones, start=1):
        story += [Paragraph(f"4.{i} {titulo}", H2)]
        for letra, txt in zip("abcdefghij", items):
            story += [Paragraph(f"4.{i}.{letra}) {txt}", BODY)]
    story += [Spacer(1, 0.3*cm),
              Paragraph("Las exclusiones recogidas en la presente sección han sido específicamente "
                        "aceptadas por el Tomador mediante su firma en las Condiciones Particulares, "
                        "conforme al artículo 3 de la Ley 50/1980, de Contrato de Seguro, que exige "
                        "que las cláusulas limitativas de los derechos del Asegurado se destaquen de "
                        "modo especial y sean expresamente aceptadas por escrito.", SMALL)]

    # ── 5. Franquicias (otra tabla) ──
    story += [PageBreak(), Paragraph("5. FRANQUICIAS", H1),
              Paragraph("Se aplicarán las siguientes franquicias por modalidad:", BODY),
              Spacer(1, 0.3*cm),
              _tabla([["Modalidad", "Franquicia general", "Franquicia específica"]] + franquicias,
                     [6*cm, 4.75*cm, 4.75*cm])]

    # ── 6. Siniestros ──
    story += [PageBreak(), Paragraph("6. DECLARACIÓN Y TRAMITACIÓN DE SINIESTROS", H1),
              Paragraph("6.1 Plazo de comunicación", H3),
              Paragraph("El Tomador deberá comunicar el siniestro al Asegurador en el plazo máximo de "
                        "<b>siete (7) días</b> desde que tuviera conocimiento del mismo.", BODY),
              Paragraph("6.2 Documentación exigible", H3),
              Paragraph("Deberá aportarse: declaración del siniestro, acreditación de la titularidad "
                        "del bien, presupuesto o factura de reparación y, cuando proceda, atestado.", BODY),
              Paragraph("6.3 Peritación", H3),
              Paragraph("En caso de desacuerdo sobre la valoración, cada parte designará un perito. De "
                        "persistir la discrepancia, se designará un tercer perito de común acuerdo.", BODY),
              Paragraph("6.4 Pago de la indemnización", H3),
              Paragraph("El Asegurador abonará la indemnización en el plazo de cuarenta (40) días desde "
                        "la recepción de la declaración del siniestro.", BODY)]

    # ── 7. Prima ──
    story += [PageBreak(), Paragraph("7. PRIMA, DURACIÓN Y RENOVACIÓN", H1),
              Paragraph("7.1 Pago de la prima", H3),
              Paragraph("La prima es anual y pagadera por anticipado.", BODY),
              Paragraph("7.2 Impago y suspensión", H3),
              Paragraph("En caso de impago de la segunda o sucesivas primas, la cobertura quedará "
                        "<b>suspendida un mes después</b> del día de su vencimiento.", BODY),
              Paragraph("7.3 Duración y prórroga", H3),
              Paragraph("El contrato se prorrogará tácitamente por periodos anuales, salvo oposición "
                        "notificada con <b>un (1) mes</b> de antelación por el Tomador o <b>dos (2) "
                        "meses</b> por el Asegurador.", BODY)]

    PolizaDoc(path, producto, codigo).multiBuild(story)
    return path

print("✅ Constructor de pólizas definido")

▶️ **Qué hace esta celda:** el catálogo de 3 productos y la generación de los PDF.

In [ ]:
import os

POLIZAS = [
    dict(
        path="HOGAR_PLUS.pdf", producto="Hogar Plus", codigo="HP-2026-01", version="3.2",
        coberturas=[
            ["Incendio, rayo y explosión", "300.000 €", "Sin franquicia"],
            ["Daños por agua por rotura accidental de conducciones", "50.000 €", "150 €"],
            ["Robo y expoliación en el interior de la vivienda", "30.000 €", "150 €"],
            ["Rotura de cristales y vitrocerámica", "3.000 €", "Sin franquicia"],
            ["Responsabilidad civil familiar", "150.000 €", "300 €"],
            ["Fenómenos atmosféricos (viento, pedrisco, nieve)", "100.000 €", "300 €"],
        ],
        exclusiones=[
            ("Daños por agua no cubiertos", [
                "Los daños causados por <b>humedades, condensación o filtraciones</b> a través de muros, "
                "fachadas, terrazas o cubiertas, aun cuando sean consecuencia de lluvia, nieve o granizo.",
                "Los daños derivados de <b>falta de mantenimiento</b> de las conducciones, así como la "
                "corrosión, el óxido o el desgaste paulatino de tuberías.",
                "El coste de <b>localización y reparación de la avería</b> cuando no se haya producido "
                "daño material indemnizable.",
                "Los daños por <b>agua de lluvia que penetre por ventanas, puertas o huecos dejados "
                "abiertos</b> o defectuosamente cerrados por el Asegurado.",
            ]),
            ("Exclusiones generales", [
                "Los daños causados con dolo o culpa grave del Asegurado.",
                "Los daños derivados de <b>vicio propio o defecto de construcción</b> preexistente.",
                "Los daños calificados como catástrofe nacional o cubiertos por el <b>Consorcio de "
                "Compensación de Seguros</b>.",
                "Los daños en <b>viviendas deshabitadas</b> más de 60 días consecutivos.",
            ]),
        ],
        franquicias=[["Vivienda habitual", "150 €", "300 € en RC familiar"],
                     ["Segunda residencia", "300 €", "600 € en daños por agua"],
                     ["Vivienda en alquiler", "300 €", "600 € en robo"]],
    ),
    dict(
        path="AUTO_TODO_RIESGO.pdf", producto="Auto Todo Riesgo", codigo="AT-2026-04", version="2.1",
        coberturas=[
            ["Responsabilidad civil obligatoria", "Ilimitada (legal)", "Sin franquicia"],
            ["Daños propios por colisión o vuelco", "Valor venal + 20%", "300 €"],
            ["Robo total o parcial del vehículo", "Valor venal", "300 €"],
            ["Incendio del vehículo", "Valor venal", "Sin franquicia"],
            ["Lunas (parabrisas, laterales y trasera)", "Sin límite", "Sin franquicia"],
            ["Asistencia en viaje desde kilómetro 0", "Incluida", "Sin franquicia"],
        ],
        exclusiones=[
            ("Circunstancias del conductor", [
                "Siniestros conduciendo bajo <b>influencia de bebidas alcohólicas</b>, drogas o estupefacientes.",
                "Siniestros cuando el conductor <b>carezca de permiso de conducción</b> en vigor.",
                "Siniestros en <b>carreras, apuestas o pruebas deportivas</b> y sus entrenamientos.",
            ]),
            ("Uso del vehículo", [
                "El uso como <b>autoescuela, alquiler sin conductor, taxi o VTC</b>, salvo declaración expresa.",
                "El transporte de <b>mercancías peligrosas</b> o de más ocupantes de los autorizados.",
                "Los daños circulando por <b>vías no aptas</b> para la circulación o fuera de calzada.",
            ]),
            ("Daños no indemnizables", [
                "El <b>desgaste, uso o defecto de conservación</b> de las piezas.",
                "Los daños <b>exclusivamente estéticos</b> que no afecten a la seguridad.",
                "La <b>depreciación</b> del vehículo tras la reparación.",
            ]),
        ],
        franquicias=[["Conductor > 25 años y > 2 años de carné", "300 €", "Sin franquicia en lunas"],
                     ["Conductor novel (< 2 años de carné)", "600 €", "600 € en daños propios"],
                     ["Conductor ocasional no declarado", "900 €", "900 € en daños propios"]],
    ),
    dict(
        path="SALUD_FAMILIAR.pdf", producto="Salud Familiar", codigo="SF-2026-02", version="1.4",
        coberturas=[
            ["Medicina primaria y especialidades", "Sin límite", "Sin franquicia"],
            ["Pruebas diagnósticas (analítica, radiología)", "Sin límite", "Sin franquicia"],
            ["Hospitalización y cirugía en centros concertados", "Sin límite", "Sin franquicia"],
            ["Urgencias 24 h en cuadro médico", "Sin límite", "Sin franquicia"],
            ["Fisioterapia y rehabilitación", "30 sesiones/año", "10 € por sesión"],
            ["Psicología clínica", "20 sesiones/año", "15 € por sesión"],
        ],
        exclusiones=[
            ("Periodos de carencia", [
                "Las <b>intervenciones quirúrgicas</b> tienen una carencia de <b>seis (6) meses</b>.",
                "El <b>parto y la asistencia al embarazo</b> tienen una carencia de <b>diez (10) meses</b>.",
                "Los <b>tratamientos de reproducción asistida</b> tienen una carencia de <b>veinticuatro "
                "(24) meses</b> y se limitan a tres ciclos.",
            ]),
            ("Prestaciones no cubiertas", [
                "Las <b>enfermedades preexistentes</b> no declaradas en el cuestionario de salud.",
                "La <b>cirugía estética</b> y todo tratamiento sin finalidad terapéutica.",
                "Los tratamientos de <b>odontología</b> salvo extracción y limpieza anual.",
                "Los <b>medicamentos y prótesis</b> no incluidos en el catálogo.",
                "La asistencia <b>fuera del cuadro médico</b>, salvo urgencia vital acreditada.",
            ]),
        ],
        franquicias=[["Modalidad sin copago", "Sin franquicia", "Sin franquicia"],
                     ["Modalidad con copago", "Según acto médico", "10 € consulta / 25 € urgencia"],
                     ["Modalidad reembolso", "20% del gasto", "Límite 60.000 €/año"]],
    ),
]

os.makedirs("polizas", exist_ok=True)
for p in POLIZAS:
    construir_poliza(os.path.join("polizas", p["path"]), p["producto"], p["codigo"],
                     p["version"], p["coberturas"], p["exclusiones"], p["franquicias"])
print(f"✅ {len(POLIZAS)} pólizas generadas en ./polizas/")

▶️ **Qué hace esta celda:** cuenta las páginas. Aquí importa por otro motivo que en el curso anterior: **cada página será una imagen que el OCR tendrá que procesar**, y en una T4 eso lleva su tiempo.

In [ ]:
from pypdf import PdfReader
import os

total = 0
for p in POLIZAS:
    n = len(PdfReader(os.path.join("polizas", p["path"])).pages)
    total += n
    print(f"{p['path']:24s} · {n:2d} páginas")
print(f"\n📄 Total: {total} páginas que pasarán por el OCR (≈ {total} imágenes).")

> 🚩 **CHECKPOINT 2** — 3 pólizas de ~8 páginas, ~24 páginas en total.

---
# 2 · Unlimited-OCR: del PDF (imagen) a markdown ⏱️ ~20 min

Aquí está el corazón del cambio. En vez de una llamada SQL a Document AI, cargamos un **modelo de visión-lenguaje** en la GPU y le pedimos que *lea* cada página.

**Tres pasos:**
1. Cargar el modelo (`baidu/Unlimited-OCR`, ~6,7 GB — se descarga una vez).
2. **Rasterizar** cada PDF a imágenes (una por página). Sí: convertimos el PDF en fotos. Así tratamos el documento **como si fuera un escaneo** — que es justo el caso donde un OCR gana a `pypdf`, y el que dejamos pendiente en el curso anterior.
3. Pasar las imágenes por el modelo → obtenemos **markdown** por página.

> 🧠 **Por qué "OCR" y no `pypdf`:** un modelo como este no "extrae caracteres", **entiende la página**: reconstruye el orden de lectura, separa columnas, detecta tablas y —clave para nosotros— marca los **títulos** como encabezados markdown. Es la versión open-source de lo que hacía el Layout Parser.

▶️ **Qué hace esta celda:** carga el modelo en la GPU. La **primera vez descarga ~6,7 GB** (unos minutos). Usamos la ruta oficial de `transformers`; **no** activamos flash-attention (el modelo no lo necesita: usa *scaled dot-product attention*, que ya viene en PyTorch).

In [ ]:
from transformers import AutoModel, AutoTokenizer

MODEL_OCR = "baidu/Unlimited-OCR"
print("⏳ Descargando y cargando el modelo (~6,7 GB la primera vez)...")
ocr_tok = AutoTokenizer.from_pretrained(MODEL_OCR, trust_remote_code=True)
ocr_model = AutoModel.from_pretrained(
    MODEL_OCR,
    trust_remote_code=True,      # el modelo trae su propio código de inferencia
    use_safetensors=True,
    torch_dtype=torch.bfloat16,  # sin attn_implementation → usa SDPA/eager (no flash-attn)
).eval().cuda()
print("✅ Unlimited-OCR cargado en la GPU")

▶️ **Qué hace esta celda:** convierte un PDF en imágenes PNG, una por página, con **PyMuPDF** (`import fitz`). Rasterizamos a 200 DPI: suficiente para leer el texto y más ligero para la GPU que 300. Mostramos la página de coberturas para ver que es, literalmente, una **foto** de la página.

In [ ]:
import fitz  # PyMuPDF
from IPython.display import Image as IPImage

def pdf_a_imagenes(pdf_path, dpi=200):
    """Rasteriza cada página del PDF a PNG. Tratamos el documento como un escaneo."""
    doc = fitz.open(pdf_path)
    carpeta = pdf_path.replace(".pdf", "_imgs")
    os.makedirs(carpeta, exist_ok=True)
    mat = fitz.Matrix(dpi / 72, dpi / 72)
    rutas = []
    for i, page in enumerate(doc):
        ruta = f"{carpeta}/p{i+1:03d}.png"
        page.get_pixmap(matrix=mat).save(ruta)
        rutas.append(ruta)
    doc.close()
    return rutas

imgs_demo = pdf_a_imagenes("polizas/HOGAR_PLUS.pdf")
print(f"✅ {len(imgs_demo)} páginas rasterizadas de Hogar Plus")
IPImage(imgs_demo[3], width=360)   # la página 4 = COBERTURAS, ahora como imagen

▶️ **Qué hace esta celda:** la función de OCR. Pasa todas las páginas de un PDF por el modelo (`infer_multi`) y recoge el resultado. Dos detalles verificados en el código del modelo:
- guarda el texto en `result.md` (markdown ya limpio, sin coordenadas — las cajas van a otro fichero);
- en multipágina **concatena las páginas con el marcador `<PAGE>`**, así que partimos por ahí para saber **qué texto es de qué página** (nuestra futura cita).

⏳ Esto es lo lento del cuaderno: **varios minutos por póliza en T4**. Momento ☕.

In [ ]:
import shutil

def ocr_documento(pdf_path):
    """Rasteriza el PDF, lo pasa por Unlimited-OCR y devuelve una lista: markdown por página."""
    imgs = pdf_a_imagenes(pdf_path, dpi=200)
    out = pdf_path.replace(".pdf", "_ocr")
    if os.path.exists(out):
        shutil.rmtree(out)
    os.makedirs(out, exist_ok=True)
    ocr_model.infer_multi(
        ocr_tok,
        prompt="<image>Multi page parsing.",   # modo documento multipágina
        image_files=imgs,
        output_path=out,
        image_size=1024,                        # multipágina usa modo 'base'
        max_length=8192,
        no_repeat_ngram_size=35, ngram_window=1024,
        save_results=True,
    )
    md_texto = open(f"{out}/result.md", encoding="utf-8").read()
    # infer_multi separa las páginas con el marcador <PAGE>
    return [p.strip() for p in md_texto.split("<PAGE>") if p.strip()]

ocr_por_doc = {}
for p in POLIZAS:
    nombre = p["path"].replace(".pdf", "")
    print(f"⏳ OCR de {nombre} ...")
    ocr_por_doc[nombre] = ocr_documento(os.path.join("polizas", p["path"]))
    print(f"   ✅ {len(ocr_por_doc[nombre])} páginas leídas")

▶️ **Qué hace esta celda:** enseña **lo que el modelo leyó** en la página de coberturas. Míralo con atención: es texto **markdown**, con su encabezado (`# 3. COBERTURAS` o similar) y la tabla. Este es el material con el que trabajaremos.

> ⚠️ **Inspecciona siempre la salida de un modelo abierto.** A diferencia de una API gestionada, aquí el formato exacto (cómo marca los títulos, si la tabla sale en HTML o markdown) puede variar. Por eso lo primero es *mirar* — y adaptar el troceado del bloque 3 a lo que veas.

In [ ]:
print("═══ Markdown de la página de COBERTURAS (Hogar Plus) ═══\n")
print(ocr_por_doc["HOGAR_PLUS"][3][:1600])

> 🚩 **CHECKPOINT 3** — Deberías ver texto legible y estructurado (encabezados, la tabla de garantías). Si ves texto ilegible o vacío: revisa que la GPU esté activa y que el modelo cargara bien. Si los títulos no salen como `#`, no pasa nada: el chunking del bloque 3 también los detecta por su numeración (`3. COBERTURAS`).

---
# 3 · De markdown a chunks con estructura ⏱️ ~15 min

Aquí aparece la **diferencia real** con el curso anterior. Document AI te devolvía los chunks **ya troceados y con su heading** (`include_ancestor_headings`). Ahora tenemos un markdown por página y **el troceado es cosa nuestra**.

No es un castigo: es control. Vamos a trocear **respetando los encabezados** del markdown y **anteponiendo a cada chunk su sección** — reproduciendo a mano justo lo que Document AI hacía por debajo. Y rastreamos la **página** (la sabemos: es el índice del `<PAGE>`).

> 🎯 El objetivo es el mismo gotcha de siempre: que el fragmento de la página de exclusiones llegue al buscador **con la etiqueta `EXCLUSIONES` pegada**, para que el RAG no diga que algo está cubierto cuando no lo está.

▶️ **Qué hace esta celda:** el troceador. Recorre el markdown de cada página, detecta encabezados (por `#` de markdown **o** por el patrón de sección `4. EXCLUSIONES`), y arma chunks anteponiendo el título de la sección vigente. Produce exactamente las columnas que espera el resto del pipeline: `documento, chunk_id, pagina_inicio, pagina_fin, heading, content`.

In [ ]:
import re
import pandas as pd

def chunkear_markdown(paginas_md, documento, max_chars=900):
    """Trocea el markdown del OCR respetando los encabezados y anteponiendo a cada
    chunk su sección (lo que include_ancestor_headings hacía automáticamente)."""
    filas, heading, n = [], "", 0
    # encabezado = línea markdown (# ...) o patrón de sección de póliza (4. EXCLUSIONES / 4.1 ...)
    es_heading = re.compile(r"^\s*(?:#{1,6}\s+(.+)|(\d+(?:\.\d+)*\.?\s+[A-ZÁÉÍÓÚÑ].{2,60}))\s*$")
    for pagina, texto in enumerate(paginas_md, start=1):
        for bloque in re.split(r"\n\s*\n", texto):
            bloque = bloque.strip()
            if not bloque:
                continue
            m = es_heading.match(bloque)
            if m:                                   # es un título: lo recordamos como contexto
                heading = (m.group(1) or m.group(2)).strip()
                continue
            for i in range(0, len(bloque), max_chars):
                n += 1
                trozo = bloque[i:i + max_chars]
                filas.append({
                    "documento": documento,
                    "chunk_id": f"{documento}-{n:04d}",
                    "pagina_inicio": pagina, "pagina_fin": pagina,
                    "heading": heading,
                    "content": f"# {heading}\n{trozo}" if heading else trozo,
                })
    return filas

filas = []
for nombre, paginas in ocr_por_doc.items():
    filas += chunkear_markdown(paginas, nombre)
df_chunks = pd.DataFrame(filas)

client.load_table_from_dataframe(
    df_chunks, f"{PROJECT_ID}.{DATASET}.polizas_chunks",
    bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE")).result()
print(f"✅ {len(df_chunks)} chunks cargados en polizas_chunks")
df_chunks.groupby("documento").size().to_frame("chunks")

▶️ **Qué hace esta celda:** el momento de la verdad, igual que en el curso anterior. Buscamos los chunks que hablan de **agua** en Hogar Plus y miramos **qué sección lleva pegada cada uno**.

In [ ]:
pd.set_option("display.max_colwidth", 90)
agua = df_chunks[(df_chunks.documento == "HOGAR_PLUS") &
                 (df_chunks.content.str.contains("agua", case=False))]
agua.assign(extracto=agua.content.str.replace(r"\s+", " ", regex=True).str[:120]) \
    [["pagina_inicio", "heading", "extracto"]]

> 🚩 **CHECKPOINT 4** — Deberías ver chunks sobre "agua" en **páginas distintas** y con **headings distintos**: unos bajo *coberturas*, otros bajo *exclusiones*. Si el `heading` sale vacío, tu OCR marcó los títulos de otra forma: vuelve a la última celda del bloque 2, mira el markdown real y **ajusta la regex `es_heading`** a lo que veas. **Ese ajuste es el trabajo de verdad de un pipeline con modelo abierto** — y el precio de no depender de una API.

---
# 4 · El contraste: OCR propio vs. Document AI ⏱️ ~5 min

Ya tenemos, con el modelo abierto, lo mismo que el curso anterior conseguía con Document AI: chunks con estructura, sección y página. Toca poner las dos vías cara a cara.

| Eje | Document AI (curso 2) | **Unlimited-OCR (este curso)** |
|---|---|---|
| **Dónde corre** | nube de Google | **tu GPU** — el dato no sale |
| **Coste** | $10 / 1.000 págs | gratis (tiempo de GPU) |
| **Velocidad** | segundos, escala sola | minutos en T4; necesitas gestionar la GPU |
| **Chunking** | incluido (layout-aware) | **tuyo** (más control, más trabajo) |
| **Salida** | JSON estable y documentado | markdown de un modelo: **hay que inspeccionarlo** |
| **Dependencia** | API propietaria | modelo **MIT**, portable |
| **Cuándo elegirlo** | quieres cero-infra y escala | **datos que no pueden salir**, coste alto por volumen, o evitar *lock-in* |

> 🎓 **No hay ganador universal.** Si procesas millones de páginas públicas y quieres olvidarte de la infra, Document AI brilla. Si tratas documentos confidenciales que **no pueden salir de tu perímetro**, o quieres controlar cada paso sin pagar por página, un modelo abierto en tu GPU es la respuesta. **La arquitectura correcta depende de la restricción que más te apriete** — y en seguros, muchas veces es la del dato.


---
# 5 · Embeddings y búsqueda ⏱️ ~8 min

**A partir de aquí todo es idéntico** a los cursos anteriores: los chunks ya son texto en una tabla de BigQuery. Vectorizamos, buscamos por significado, y montamos el RAG. Vamos rápido.

> 🎓 Recordatorio: `RETRIEVAL_DOCUMENT` al vectorizar los chunks, `RETRIEVAL_QUERY` al vectorizar la pregunta; `VECTOR_SEARCH` devuelve los chunks cuyo vector está más cerca del de la pregunta.

▶️ **Qué hace esta celda:** registra el modelo remoto de embeddings y vectoriza los chunks.

In [ ]:
# Modelo de embeddings (Vertex) — el mismo de siempre
for intento in range(1, 5):
    try:
        client.query(f"""
        CREATE OR REPLACE MODEL `{PROJECT_ID}.{DATASET}.embedding_model`
        REMOTE WITH CONNECTION `{PROJECT_ID}.{LOCATION.lower()}.{CONN_ID}`
        OPTIONS (ENDPOINT = 'gemini-embedding-001')
        """).result()
        print("✅ Modelo de embeddings creado")
        break
    except Exception as e:
        if intento == 4:
            raise
        print(f"⏳ Permisos propagándose ({intento}/4)... reintento en 30 s")
        time.sleep(30)

client.query(f"""
CREATE OR REPLACE TABLE `{PROJECT_ID}.{DATASET}.polizas_embeddings` AS
SELECT * FROM ML.GENERATE_EMBEDDING(
  MODEL `{PROJECT_ID}.{DATASET}.embedding_model`,
  (SELECT chunk_id, documento, pagina_inicio, pagina_fin, heading, content
   FROM `{PROJECT_ID}.{DATASET}.polizas_chunks`),
  STRUCT(TRUE AS flatten_json_output, 'RETRIEVAL_DOCUMENT' AS task_type))
WHERE ml_generate_embedding_status = ''
""").result()
print("✅ Chunks vectorizados")

▶️ **Qué hace esta celda:** la función de búsqueda semántica, con filtro opcional por póliza (búsqueda híbrida). Idéntica a la del curso anterior.

In [ ]:
def buscar_clausulas(pregunta: str, k: int = 5, documento: str | None = None):
    """Devuelve los k chunks más relevantes, con su procedencia para poder citarlos."""
    filtro = "AND base.documento = @documento" if documento else ""
    sql = f"""
    SELECT base.documento, base.pagina_inicio AS pagina, base.heading,
           base.content, ROUND(distance, 4) AS distancia
    FROM VECTOR_SEARCH(
      TABLE `{PROJECT_ID}.{DATASET}.polizas_embeddings`,
      'ml_generate_embedding_result',
      (SELECT ml_generate_embedding_result
       FROM ML.GENERATE_EMBEDDING(
         MODEL `{PROJECT_ID}.{DATASET}.embedding_model`,
         (SELECT @pregunta AS content),
         STRUCT(TRUE AS flatten_json_output, 'RETRIEVAL_QUERY' AS task_type))),
      top_k => @k, distance_type => 'COSINE')
    WHERE TRUE {filtro}
    ORDER BY distance
    """
    params = [bigquery.ScalarQueryParameter("pregunta", "STRING", pregunta),
              bigquery.ScalarQueryParameter("k", "INT64", k)]
    if documento:
        params.append(bigquery.ScalarQueryParameter("documento", "STRING", documento))
    return client.query(sql, job_config=bigquery.QueryJobConfig(query_parameters=params)).to_dataframe()

# 🎯 La pregunta del millón. Mira los HEADINGS de lo que recupera:
buscar_clausulas("¿me cubre el agua de lluvia que ha entrado en casa?",
                 k=4, documento="HOGAR_PLUS")[["documento", "pagina", "heading", "distancia"]]

> 🎓 Como en el curso anterior, el buscador recupera chunks de **coberturas y de exclusiones** a la vez: semánticamente los dos hablan de agua. La distinción la pone el **heading**, que por eso viaja hasta el LLM.

---
# 6 · RAG con citas verificables ⏱️ ~12 min

Idéntico al curso anterior: recuperamos, montamos el contexto **con la procedencia** (póliza, página, sección) y exigimos al modelo que **cite** y que distinga cobertura de exclusión.

▶️ **Qué hace esta celda:** el RAG completo. La respuesta se apoya solo en los fragmentos recuperados, cada uno con su origen.

In [ ]:
from google import genai

genai_client = genai.Client(vertexai=True, project=PROJECT_ID, location="global")

def responder(pregunta: str, k: int = 6, documento: str | None = None, verbose: bool = True) -> str:
    # 1) Retrieval
    docs = buscar_clausulas(pregunta, k, documento)

    # 2) Augmentation — cada fragmento viaja CON su procedencia
    contexto = "\n\n---\n\n".join(
        f"[Póliza: {r.documento} | Página: {r.pagina} | Sección: {r.heading}]\n{r.content}"
        for r in docs.itertuples()
    )
    prompt = f"""Eres el asistente del equipo de atención al cliente de Peñalara Seguros, S.A.
Respondes sobre lo que cubre o no cubre una póliza, usando EXCLUSIVAMENTE los fragmentos del contexto.

REGLAS OBLIGATORIAS:
1. Antes de afirmar que algo está cubierto, comprueba la SECCIÓN del fragmento.
   Un fragmento bajo "EXCLUSIONES" significa que NO está cubierto, aunque hable del mismo riesgo.
2. Si el mismo riesgo aparece en coberturas Y en exclusiones, explica el MATIZ que los separa
   (normalmente la causa del daño distingue un caso del otro).
3. Cita SIEMPRE la póliza, la página y la cláusula concreta. Formato: (Hogar Plus, pág. 5, cláusula 4.1.a)
4. Si el contexto no basta para responder, dilo claramente. NO inventes coberturas.

### Contexto (fragmentos recuperados del condicionado):
{contexto}

### Pregunta del cliente:
{pregunta}"""

    # 3) Generation
    resp = genai_client.models.generate_content(model="gemini-2.5-flash", contents=prompt)
    if verbose:
        secciones = [s for s in docs.heading.unique().tolist() if s]
        print(f"🔎 {len(docs)} fragmentos | secciones tocadas: {secciones}\n")
    return resp.text or "(respuesta vacía o bloqueada por el filtro de seguridad — reintenta)"

# 🎯 LA PREGUNTA TRAMPA — la que un RAG naive responde mal
print(responder("Ha entrado agua de lluvia por la terraza y se me ha estropeado el parqué. "
                "¿Me lo cubre el seguro de hogar?", documento="HOGAR_PLUS"))

> 🎓 Si todo fue bien, ante *"agua de lluvia por la terraza"* el modelo responde **NO cubierto**, citando la cláusula de exclusiones. Y lo hace sobre un texto que extrajo **un modelo open-source en tu propia GPU**, sin que la póliza saliera de la máquina.

▶️ **Qué hacen estas dos celdas:** más preguntas, tocando otras pólizas y matices.

In [ ]:
print(responder("Contraté el seguro de salud hace 3 meses y necesito operarme del menisco. "
                "¿Me lo cubren ya?", documento="SALUD_FAMILIAR"))

In [ ]:
print(responder("Tuve un accidente y el coche lo conducía mi sobrino, que sacó el carné hace 8 meses "
                "y no está declarado en la póliza. ¿Qué franquicia me toca pagar?",
                documento="AUTO_TODO_RIESGO"))

> 🎓 **Anti-alucinación:** prueba con `responder("¿Cubre la póliza los daños por un ataque de dragones?")`. El modelo debería reconocer que no está en el contexto en vez de inventarse una cláusula.

---
# 7 · Versionado de documentos ⏱️ ~8 min

Mismo problema que en el curso anterior (una versión nueva **sustituye** a la vieja), con un matiz propio de este pipeline: reprocesar significa **volver a pasar el PDF por el OCR**. Como el OCR es local y gratis, no hay factura por página — pero sí tiempo de GPU. Reprocesamos **solo** la póliza que cambió.

▶️ **Qué hace esta celda:** publica **Hogar Plus v3.3** (sube la franquicia de agua de 150 € a 250 €), la **vuelve a pasar por el OCR**, y reemplaza sus chunks y sus vectores (`DELETE` + recarga por documento).

In [ ]:
# 1) Nueva versión: cambia la franquicia de daños por agua
nueva = dict(POLIZAS[0]); nueva["version"] = "3.3"
nueva["coberturas"] = [c[:] for c in POLIZAS[0]["coberturas"]]
nueva["coberturas"][1][2] = "250 €"      # 150 € → 250 €
construir_poliza("polizas/HOGAR_PLUS.pdf", nueva["producto"], nueva["codigo"],
                 nueva["version"], nueva["coberturas"], nueva["exclusiones"], nueva["franquicias"])
print("✅ Hogar Plus v3.3 regenerada")

# 2) Re-OCR SOLO de esa póliza y re-chunk
print("⏳ Re-OCR de Hogar Plus v3.3...")
filas_v33 = chunkear_markdown(ocr_documento("polizas/HOGAR_PLUS.pdf"), "HOGAR_PLUS")

# 3) Fuera la versión vieja, entra la nueva (chunks y embeddings)
client.query(f"DELETE FROM `{PROJECT_ID}.{DATASET}.polizas_chunks` WHERE documento='HOGAR_PLUS'").result()
client.load_table_from_dataframe(
    pd.DataFrame(filas_v33), f"{PROJECT_ID}.{DATASET}.polizas_chunks",
    bigquery.LoadJobConfig(write_disposition="WRITE_APPEND")).result()

client.query(f"DELETE FROM `{PROJECT_ID}.{DATASET}.polizas_embeddings` WHERE documento='HOGAR_PLUS'").result()
job = client.query(f"""
INSERT INTO `{PROJECT_ID}.{DATASET}.polizas_embeddings`
SELECT * FROM ML.GENERATE_EMBEDDING(
  MODEL `{PROJECT_ID}.{DATASET}.embedding_model`,
  (SELECT chunk_id, documento, pagina_inicio, pagina_fin, heading, content
   FROM `{PROJECT_ID}.{DATASET}.polizas_chunks` WHERE documento='HOGAR_PLUS'),
  STRUCT(TRUE AS flatten_json_output, 'RETRIEVAL_DOCUMENT' AS task_type))
WHERE ml_generate_embedding_status = ''
""")
job.result()
print(f"✅ Reemplazados {job.num_dml_affected_rows} chunks de Hogar Plus (v3.2 → v3.3)")

▶️ **Qué hace esta celda:** comprueba que el RAG ya responde con la franquicia **nueva** (250 €).

In [ ]:
print(responder("¿Cuál es la franquicia por daños por agua en el seguro de hogar?",
                documento="HOGAR_PLUS"))

---
# 8 · Evaluación, ejercicios y limpieza ⏱️ ~9 min

### Mini-evaluación: ¿acierta el sentido de la respuesta?

▶️ **Qué hace esta celda:** casos con respuesta conocida, donde solo la jerarquía (heading) desempata cobertura de exclusión.

In [ ]:
casos = [
    ("Se ha roto una tubería del baño y ha inundado el salón. ¿Está cubierto?",
     "HOGAR_PLUS", "SÍ (cláusula 3.2, rotura accidental de conducciones)"),
    ("Entra agua de lluvia por una filtración en la terraza. ¿Está cubierto?",
     "HOGAR_PLUS", "NO (cláusula 4.1.a, filtraciones aunque sean por lluvia)"),
    ("Dejé la ventana abierta, llovió y se estropeó el suelo. ¿Está cubierto?",
     "HOGAR_PLUS", "NO (cláusula 4.1.d, agua por huecos dejados abiertos)"),
    ("Necesito una operación de rodilla a los 8 meses de contratar. ¿Cubierta?",
     "SALUD_FAMILIAR", "SÍ (carencia quirúrgica de 6 meses ya superada)"),
    ("Quiero una rinoplastia estética. ¿La cubre el seguro de salud?",
     "SALUD_FAMILIAR", "NO (exclusión: cirugía estética sin fin terapéutico)"),
]

for pregunta, doc, esperado in casos:
    print("═" * 78)
    print(f"❓ {pregunta}")
    print(f"🎯 Esperado: {esperado}")
    print(f"🤖 {responder(pregunta, documento=doc, verbose=False)[:300]}...\n")

### 🧪 Autoevaluación — cinco preguntas

1. ¿Por qué rasterizamos el PDF a imágenes antes del OCR, si ya tenía texto?
2. En este curso el `heading` de cada chunk lo pusimos nosotros. ¿De dónde salía en el curso de Document AI?
3. ¿Qué gana y qué pierde una empresa al cambiar Document AI por un modelo open-source en su GPU?
4. El OCR marcó los títulos de una forma inesperada y tus `heading` salen vacíos. ¿Dónde miras y qué tocas?
5. ¿Por qué el resto del pipeline (embeddings, `VECTOR_SEARCH`, RAG) no cambió ni una línea entre los dos cursos?

### Ejercicios para casa

- **Fácil** — Cambia `dpi=200` a `dpi=300` en `pdf_a_imagenes` y mira si mejora la calidad del markdown (y cuánto más tarda).
- **Medio** — Sustituye una póliza por un **PDF realmente escaneado** (una foto de un papel) y comprueba que el OCR lo lee igual, mientras que `pypdf` devolvería basura.
- **Medio** — Añade a `chunkear_markdown` la conservación de la **jerarquía completa** (sección + subsección, p. ej. `4 › 4.1`), no solo el último título.
- **Difícil** — Compara chunk a chunk el texto del OCR con el texto real del PDF (`pypdf`) y mide la **tasa de error** del OCR sobre nuestras pólizas. ¿Dónde falla más: tablas, letra pequeña, números?
- **Difícil** — Monta el OCR como un **servicio** (FastAPI) que reciba un PDF y devuelva el markdown, para separar la GPU del resto del pipeline.

### Ideas para producción

- **La GPU es el cuello de botella**: procesa por lotes, en horas valle, y cachea por `md5` del PDF para no re-OCR-ear lo que no cambió.
- **Inspecciona y valida el markdown** automáticamente (¿tiene los headings esperados? ¿el nº de páginas coincide?) antes de dejarlo entrar al índice.
- **vLLM / SGLang** (el modelo los soporta) sirven el OCR con mucho más throughput que la ruta `transformers` de este cuaderno.

### 🧹 Limpieza de recursos

In [ ]:
BORRAR = False  # ⚠️ cambia a True para borrar el dataset y la conexión

if BORRAR:
    client.delete_dataset(f"{PROJECT_ID}.{DATASET}", delete_contents=True, not_found_ok=True)
    print("🗑️  Dataset borrado")
    r = subprocess.run(["bq", "rm", "--connection", "--force",
                        f"{PROJECT_ID}.{LOCATION}.{CONN_ID}"], capture_output=True, text=True)
    print("🗑️  Conexión borrada" if r.returncode == 0 else "ℹ️  Conexión ya no existía")
else:
    print("ℹ️  BORRAR = False. (No hay bucket ni processor que borrar: la extracción fue local.)")

# Liberamos la GPU en cualquier caso
import gc
try:
    del ocr_model, ocr_tok
    gc.collect(); torch.cuda.empty_cache()
    print("🧹 Modelo descargado y GPU liberada")
except NameError:
    pass

---
## 🧭 Mapa final: qué te llevas

1. **La extracción es una decisión de arquitectura, no un detalle.** API gestionada vs. modelo propio no es "mejor o peor": es *dónde vive tu dato* y *quién controla el pipeline*.
2. **Un modelo abierto te da soberanía y coste cero por página, a cambio de infra y trabajo de parsing.** El markdown hay que inspeccionarlo y el chunking es tuyo.
3. **El OCR moderno entiende el layout**, no solo lee caracteres: por eso podemos reconstruir headings y tablas y mantener la jerarquía que salva el RAG.
4. **El pipeline de conocimiento (embeddings → búsqueda → RAG) es independiente de cómo extraigas el texto.** Cambiamos Document AI por Unlimited-OCR y de la sección 5 en adelante no tocamos ni una línea.

### 📚 Para seguir
- [Unlimited-OCR (Hugging Face)](https://huggingface.co/baidu/Unlimited-OCR) · [código en GitHub](https://github.com/baidu/Unlimited-OCR)
- [DeepSeek-OCR](https://github.com/deepseek-ai/DeepSeek-OCR), del que hereda la arquitectura
- Los cursos anteriores: [RAG sobre emails](https://colab.research.google.com/github/noelserdna/colab-gcp-ia/blob/main/curso_rag_emails_bigquery_v2.ipynb) · [RAG sobre PDF con Document AI](https://colab.research.google.com/github/noelserdna/colab-gcp-ia/blob/main/curso_rag_pdf_polizas_bigquery.ipynb)
